In [9]:

#### Calculated climos from history output


In [10]:
import xarray as xr
import numpy as np
import pandas as pd
#import xcdat as xcd

import glob as glob
import os as os
import re as re
import cftime as cft

In [11]:
from dask.distributed import Client
from dask_jobqueue import PBSCluster

In [12]:
cluster = PBSCluster(
    account="P03010039",
    interface="ext",
    walltime="04:00:00",
    queue="main",
    cores=4,            # threads per worker (reductions scale well on threads)
    processes=1,        # 1 worker per PBS job → bigger memory pool, less shuffle
    memory="32GB",
    local_directory="/glade/derecho/scratch/rneale/dask-temp",
    log_directory="/glade/derecho/scratch/rneale/dask-logs",
)

cluster.scale(jobs=12)   # 12 workers × 4 threads = 48 compute threads
client = Client(cluster)

client

## Setup Run Information

/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42435 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: /node/crhtc67.hpc.ucar.edu/10107/proxy/42435/status,
Dashboard: /node/crhtc67.hpc.ucar.edu/10107/proxy/42435/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.175:45571,Workers: 0
Dashboard: /node/crhtc67.hpc.ucar.edu/10107/proxy/42435/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [15]:
''' Case Details '''

import sys

#run_name = 'f.cam6_3_161.FLTHIST_ne30.ke.001'

#run_name = 'f.e22.FHIST.f09_f09.CAM6.L32.000'
run_name = 'f.e30.FLTHIST.CAM7.L48.000a'

#run_name = 'f.e30.FHISTC_WAt1ma.ne30pg3_mg17_L135_cam6_4_173_beres0.15_num_cin1'

dir0_in = '/glade/derecho/scratch/rneale/archive/'
#dir0_in = '/glade/campaign/acom/acom-climate/mijeong/archive/'
#dir0_in = '/glade/derecho/scratch/hannay/archive/'
#dir0_in = '/glade/derecho/scratch/gmarques/archive/'
#dir0_in = '/glade/derecho/scratch/aherring/archive/'


dir0_out = '/glade/derecho/scratch/rneale/archive/'

# History tape prefix (change this one place to switch between h0, h0a, h1, …)
hist_pref = '.h0a.'
# Short label (e.g. 'h0', 'h0a') derived from hist_pref for messages/regex.
hist_tag  = hist_pref.strip('.')

years = [1979, 2005]
lt_dim = True   # Add in the single value time dim for compatibility with other scripts

########################################

dir_in  = dir0_in  + run_name + '/atm/hist/'
dir_out = dir0_out + run_name + '/climo/'

syears = [str(y) for y in years]   # ['1979', '2005']

# Create dir if needed.
if not os.path.exists(dir_out):
    os.makedirs(dir_out)
    print(f"Directory created: {dir_out}")
else:
    print(f"Directory already exists: {dir_out}")


# ── Variables to drop at open_mfdataset time (never read, never graphed) ──
# Aerosol vars (both _a and _c suites that aren't needed downstream)
drop_vars_aer = [
    'bc_c1','bc_c4','dst_c1','dst_c2','dst_c3','ncl_c1','ncl_c2','ncl_c3',
    'num_c1','num_c2','num_c3','num_c4','pom_c1','pom_c4',
    'so4_c1','so4_c2','so4_c3','soa_c1','soa_c2',
    'bc_a1','bc_a4','dst_a1','dst_a2','dst_a3','ncl_a1','ncl_a2','ncl_a3',
    'num_a1','num_a2','num_a3','so4_a1','so4_a2','so4_a3','soa_a1','soa_a2',
]
# Other unwanted 3D vars
drop_vars_3d = [
    'ADRAIN','ADSNOW','ANSNOW','AWNI','AREI','AREL','AWNC','CCN3','CFC11','CFC12',
    'CH4','CO2','DMS','GRAUQM','H2O2','H2SO4','N2O','NUMGRA','SNOWQM','SO2','SOAE','SOAG',
]
# String/time-less helpers that don't survive a groupby
drop_vars_xtime = ['date_written', 'time_written', 'trop_cld_lev']

DROP_AT_OPEN = drop_vars_aer + drop_vars_3d + drop_vars_xtime


# ── Sort files ─────────────────────────────────────────────────────────────
files_unsorted = os.path.join(dir_in, run_name + '*' + hist_pref + '*.nc')
print(files_unsorted)
files_sorted = sorted(glob.glob(files_unsorted))
print(f'-List of all {hist_tag}* files')
print(files_sorted[0])
print(files_sorted[-1])
print()
print(f'-List of all files for requested years - {syears[0]} to {syears[1]}')

# Tight regex: match YYYY from the "{hist_pref}YYYY-MM.nc" suffix.
# re.escape(hist_pref) handles the leading/trailing dots for us.
pattern = re.compile(re.escape(hist_pref) + r'(\d{4})-\d{2}\.nc$')
files_in = []
for ff in files_sorted:
    m = pattern.search(ff)
    if m is None:
        continue
    year = int(m.group(1))
    if years[0] <= year <= years[1]:
        files_in.append(ff)

print(files_in[0])
print(files_in[-1])

# File number check
print()
expected = 12 * (years[1] - years[0] + 1)
if len(files_in) != expected:
    msg = (f'INCORRECT NUMBER OF FILES FOR YEARS REQUESTED -- '
           f'{years[0]} to {years[1]}: found {len(files_in)}, expected {expected}')
    print(msg)
    raise RuntimeError(msg)
else:
    print(f'CORRECT NUMBER OF FILES FOR YEARS REQUESTED -- {years[0]} to {years[1]}')

Directory already exists: /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/climo/
/glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/atm/hist/f.e30.FLTHIST.CAM7.L48.000a*.h0a.*.nc
-List of all h0a* files
/glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/atm/hist/f.e30.FLTHIST.CAM7.L48.000a.cam.h0a.1979-01.nc
/glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/atm/hist/f.e30.FLTHIST.CAM7.L48.000a.cam.h0a.2008-12.nc

-List of all files for requested years - 1979 to 2005
/glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/atm/hist/f.e30.FLTHIST.CAM7.L48.000a.cam.h0a.1979-01.nc
/glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/atm/hist/f.e30.FLTHIST.CAM7.L48.000a.cam.h0a.2005-12.nc

CORRECT NUMBER OF FILES FOR YEARS REQUESTED -- 1979 to 2005


In [16]:
'''
    Read in data (lazy).

    Keys for a small task graph + a chunking that matches the reduction axis:
      - drop_variables  → aerosols / 3-D CFC-CO2 are never opened or graphed
      - data_vars='minimal', coords='minimal', compat='override' → skip
        per-var metadata reconciliation across 324 files
      - chunks={'time': 60} → ~5 chunks total per var over 324 months,
        with whole spatial dims (lat/lon/lev kept together for the groupby)
'''
ds_hist = xr.open_mfdataset(
    files_in,
    parallel=True,
    chunks={'time': 48, 'lat': -1, 'lon': -1, 'lev': -1},
    drop_variables=DROP_AT_OPEN,
    data_vars='minimal', coords='minimal', compat='override',
)

In [17]:

ds_hist.time

<xarray.DataArray 'time' (time: 322)> Size: 3kB
array([cftime.DatetimeNoLeap(1979, 1, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1979, 2, 15, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1979, 3, 16, 12, 0, 0, 0, has_year_zero=True),
       ...,
       cftime.DatetimeNoLeap(2005, 10, 16, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2005, 11, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2005, 12, 16, 12, 0, 0, 0, has_year_zero=True)],
      dtype=object)
Coordinates:
  * time     (time) object 3kB 1979-01-16 12:00:00 ... 2005-12-16 12:00:00
Attributes:
    long_name:  time
    bounds:     time_bounds

In [18]:
start_time = cft.DatetimeNoLeap(years[0], 1, 1)
end_time = cft.DatetimeNoLeap(years[1], 12, 31)

ds_hist = ds_hist.sel(time=slice(start_time, end_time))

In [19]:
'''
    Prepare ds_vars for the reduction:
      - drop_variables=DROP_AT_OPEN already handled the big vars at open time
      - collapse time-invariant 'constants' (hyam/hybm/hyai/hybi/P0/gw) to a
        single time slice (if they still carry a time dim after open_mfdataset)
      - cast once to float32 (halves memory for the whole pipeline)
'''

# Collapse time-invariant variables if open_mfdataset gave them a time dim.
# (With data_vars='minimal' they usually don't, but be defensive.)
CONSTS = ['hyam', 'hybm', 'hyai', 'hybi', 'P0', 'gw']
ds_vars = ds_hist
for v in CONSTS:
    if v in ds_vars.variables and 'time' in ds_vars[v].dims:
        ds_vars = ds_vars.assign({v: ds_vars[v].isel(time=0, drop=True)})

# Cast prognostic/physics vars to float32 once (constants stay float64).
# This halves memory across all downstream intermediates.
float_vars = [v for v in ds_vars.data_vars
              if v not in CONSTS and ds_vars[v].dtype == np.float64]
if float_vars:
    ds_vars = ds_vars.assign(**{v: ds_vars[v].astype(np.float32) for v in float_vars})

# Provenance attrs
ds_vars.attrs['History Directory'] = dir_in
ds_vars.attrs['Start Year']        = str(years[0])
ds_vars.attrs['End Year']          = str(years[1])

ds_vars

<xarray.Dataset> Size: 318GB
Dimensions:           (lat: 192, lev: 48, ilev: 49, time: 322, nbnd: 2,
                       lon: 288, trop_pref: 48, trop_prefi: 49)
Coordinates:
  * lat               (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon               (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.3 357.5 358.8
  * lev               (lev) float64 384B 3.063 5.244 8.821 ... 957.5 976.3 992.6
  * ilev              (ilev) float64 392B 2.313 3.813 6.674 ... 985.1 1e+03
  * trop_pref         (trop_pref) float64 384B 3.063 5.244 8.821 ... 976.3 992.6
  * trop_prefi        (trop_prefi) float64 392B 2.313 3.813 ... 985.1 1e+03
  * time              (time) object 3kB 1979-01-16 12:00:00 ... 2005-12-16 12...
Dimensions without coordinates: nbnd
Data variables: (12/322)
    w                 (lat) float32 768B dask.array<chunksize=(192,), meta=np.ndarray>
    hyam              (lev) float64 384B dask.array<chunksize=(48,), meta=np.ndarray>
    hybm              (lev) float64 384B dask.array<chunksize=(48,), meta=np.ndarray>
    hyai              (ilev) float64 392B dask.array<chunksize=(49,), meta=np.ndarray>
    hybi              (ilev) float64 392B dask.array<chunksize=(49,), meta=np.ndarray>
    date              (time) int32 1kB dask.array<chunksize=(1,), meta=np.ndarray>
    ...                ...
    RTP2_CLUBB        (time, ilev, lat, lon) float32 3GB dask.array<chunksize=(1, 49, 192, 288), meta=np.ndarray>
    THLP2_CLUBB       (time, ilev, lat, lon) float32 3GB dask.array<chunksize=(1, 49, 192, 288), meta=np.ndarray>
    RTPTHLP_CLUBB     (time, ilev, lat, lon) float32 3GB dask.array<chunksize=(1, 49, 192, 288), meta=np.ndarray>
    WPRCP_CLUBB       (time, ilev, lat, lon) float32 3GB dask.array<chunksize=(1, 49, 192, 288), meta=np.ndarray>
    WPTHVP_CLUBB      (time, ilev, lat, lon) float32 3GB dask.array<chunksize=(1, 49, 192, 288), meta=np.ndarray>
    ZM_CLUBB          (time, ilev, lat, lon) float32 3GB dask.array<chunksize=(1, 49, 192, 288), meta=np.ndarray>
Attributes: (12/14)
    interp_type:        bilinear
    interp_outputgri:   equally spaced with poles
    Conventions:        CF-1.0
    source:             CAM
    case:               f.e30.FLTHIST.CAM7.L48.000a
    logname:            rneale
    ...                 ...
    topography_file:    /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/...
    model_doi_url:      not_set
    time_period_freq:   month_1
    History Directory:  /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.C...
    Start Year:         1979
    End Year:           2005

In [20]:
'''
 Calculate monthly climo - persist to worker memory once; annual and seasonal
 derive from this (no extra I/O).
'''
from dask.distributed import wait

ds_cmonth_base = ds_vars.groupby('time.month').mean(keep_attrs=True)

print('Computing monthly climatology (single full pass over data)...')
ds_cmonth_base = ds_cmonth_base.persist()   # stays on workers, never pulled to kernel
wait(ds_cmonth_base)                         # block until workers finish
print('-Done.')

# Month names for output file.
mname_file = ['%02d' % x for x in ds_cmonth_base.month.values]

# Add the trailing single-value time dimension for compatibility.
ds_cmonth = ds_cmonth_base.expand_dims(time=[ds_vars.time.values[0]])
ds_cmonth.coords['time'].attrs = ds_vars.coords['time'].attrs
ds_cmonth

Computing monthly climatology (single full pass over data)...


/glade/u/apps/opt/conda/envs/npl-2024b/lib/python3.11/site-packages/distributed/client.py:3245: UserWarning: Sending large graph of size 61.02 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


-Done.


<xarray.Dataset> Size: 12GB
Dimensions:           (time: 1, month: 12, lat: 192, lon: 288, lev: 48,
                       ilev: 49, trop_pref: 48, trop_prefi: 49)
Coordinates:
  * time              (time) object 8B 1979-01-16 12:00:00
  * lat               (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon               (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.3 357.5 358.8
  * lev               (lev) float64 384B 3.063 5.244 8.821 ... 957.5 976.3 992.6
  * ilev              (ilev) float64 392B 2.313 3.813 6.674 ... 985.1 1e+03
  * trop_pref         (trop_pref) float64 384B 3.063 5.244 8.821 ... 976.3 992.6
  * trop_prefi        (trop_prefi) float64 392B 2.313 3.813 ... 985.1 1e+03
  * month             (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
Data variables: (12/321)
    date              (time, month) float64 96B dask.array<chunksize=(1, 1), meta=np.ndarray>
    datesec           (time, month) float64 96B dask.array<chunksize=(1, 1), meta=np.ndarray>
    AEROD_v           (time, month, lat, lon) float32 3MB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    AODDUST           (time, month, lat, lon) float32 3MB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    AODDUST01         (time, month, lat, lon) float32 3MB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    AODDUST03         (time, month, lat, lon) float32 3MB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    ...                ...
    hybi              (time, month, ilev) float64 5kB dask.array<chunksize=(1, 12, 49), meta=np.ndarray>
    ndbase            (time, month) int32 48B 0 0 0 0 0 0 0 0 0 0 0 0
    nsbase            (time, month) int32 48B 0 0 0 0 0 0 0 0 0 0 0 0
    nbdate            (time, month) int32 48B 19790101 19790101 ... 19790101
    nbsec             (time, month) int32 48B 0 0 0 0 0 0 0 0 0 0 0 0
    mdt               (time, month) int32 48B 1800 1800 1800 ... 1800 1800 1800
Attributes: (12/14)
    interp_type:        bilinear
    interp_outputgri:   equally spaced with poles
    Conventions:        CF-1.0
    source:             CAM
    case:               f.e30.FLTHIST.CAM7.L48.000a
    logname:            rneale
    ...                 ...
    topography_file:    /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/...
    model_doi_url:      not_set
    time_period_freq:   month_1
    History Directory:  /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.C...
    Start Year:         1979
    End Year:           2005

In [21]:
print('-Calculate annual climatology (derived from monthly climo, no extra I/O)...')

ds_cyear = ds_cmonth_base.mean(dim='month', keep_attrs=True)
ds_cyear

-Calculate annual climatology (derived from monthly climo, no extra I/O)...


<xarray.Dataset> Size: 986MB
Dimensions:           (lat: 192, lon: 288, lev: 48, ilev: 49, trop_pref: 48,
                       trop_prefi: 49)
Coordinates:
  * lat               (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon               (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.3 357.5 358.8
  * lev               (lev) float64 384B 3.063 5.244 8.821 ... 957.5 976.3 992.6
  * ilev              (ilev) float64 392B 2.313 3.813 6.674 ... 985.1 1e+03
  * trop_pref         (trop_pref) float64 384B 3.063 5.244 8.821 ... 976.3 992.6
  * trop_prefi        (trop_prefi) float64 392B 2.313 3.813 ... 985.1 1e+03
Data variables: (12/321)
    date              float64 8B dask.array<chunksize=(), meta=np.ndarray>
    datesec           float64 8B dask.array<chunksize=(), meta=np.ndarray>
    AEROD_v           (lat, lon) float32 221kB dask.array<chunksize=(192, 288), meta=np.ndarray>
    AODDUST           (lat, lon) float32 221kB dask.array<chunksize=(192, 288), meta=np.ndarray>
    AODDUST01         (lat, lon) float32 221kB dask.array<chunksize=(192, 288), meta=np.ndarray>
    AODDUST03         (lat, lon) float32 221kB dask.array<chunksize=(192, 288), meta=np.ndarray>
    ...                ...
    hybi              (ilev) float64 392B dask.array<chunksize=(49,), meta=np.ndarray>
    ndbase            float64 8B 0.0
    nsbase            float64 8B 0.0
    nbdate            float64 8B 1.979e+07
    nbsec             float64 8B 0.0
    mdt               float64 8B 1.8e+03
Attributes: (12/14)
    interp_type:        bilinear
    interp_outputgri:   equally spaced with poles
    Conventions:        CF-1.0
    source:             CAM
    case:               f.e30.FLTHIST.CAM7.L48.000a
    logname:            rneale
    ...                 ...
    topography_file:    /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/...
    model_doi_url:      not_set
    time_period_freq:   month_1
    History Directory:  /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.C...
    Start Year:         1979
    End Year:           2005

In [22]:
print('-Calculate weighted seasonal climatology (derived from monthly climo, no extra I/O)...')

_days = {1:31, 2:28, 3:31, 4:30, 5:31, 6:30, 7:31, 8:31, 9:30, 10:31, 11:30, 12:31}
_season_months = {'DJF': [12,1,2], 'MAM': [3,4,5], 'JJA': [6,7,8], 'SON': [9,10,11]}

# Build one lazy weighted sum per season.  Keep it lazy — the final concat +
# per-season to_netcdf (compute=False) in the write cell will drive one
# combined dask.compute(), avoiding the 4x duplicate persist() of before.
seas_list = []
for sname, months in _season_months.items():
    days    = np.array([_days[m] for m in months], dtype=float)
    weights = days / days.sum()

    weighted = None
    for w, m in zip(weights, months):
        term = float(w) * ds_cmonth_base.sel(month=m)
        weighted = term if weighted is None else weighted + term

    seas_list.append(weighted.expand_dims(season=[sname]))

ds_wcseas = xr.concat(seas_list, dim='season')
ds_wcseas

-Calculate weighted seasonal climatology (derived from monthly climo, no extra I/O)...


<xarray.Dataset> Size: 4GB
Dimensions:           (season: 4, lat: 192, lon: 288, lev: 48, ilev: 49,
                       trop_pref: 48, trop_prefi: 49)
Coordinates:
  * season            (season) object 32B 'DJF' 'MAM' 'JJA' 'SON'
  * lat               (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon               (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.3 357.5 358.8
  * lev               (lev) float64 384B 3.063 5.244 8.821 ... 957.5 976.3 992.6
  * ilev              (ilev) float64 392B 2.313 3.813 6.674 ... 985.1 1e+03
  * trop_pref         (trop_pref) float64 384B 3.063 5.244 8.821 ... 976.3 992.6
  * trop_prefi        (trop_prefi) float64 392B 2.313 3.813 ... 985.1 1e+03
    month             (season) int64 32B 2 5 8 11
Data variables: (12/321)
    date              (season) float64 32B dask.array<chunksize=(1,), meta=np.ndarray>
    datesec           (season) float64 32B dask.array<chunksize=(1,), meta=np.ndarray>
    AEROD_v           (season, lat, lon) float32 885kB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    AODDUST           (season, lat, lon) float32 885kB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    AODDUST01         (season, lat, lon) float32 885kB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    AODDUST03         (season, lat, lon) float32 885kB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
    ...                ...
    hybi              (season, ilev) float64 2kB dask.array<chunksize=(1, 49), meta=np.ndarray>
    ndbase            (season) float64 32B 0.0 0.0 0.0 0.0
    nsbase            (season) float64 32B 0.0 0.0 0.0 0.0
    nbdate            (season) float64 32B 1.979e+07 1.979e+07 ... 1.979e+07
    nbsec             (season) float64 32B 0.0 0.0 0.0 0.0
    mdt               (season) float64 32B 1.8e+03 1.8e+03 1.8e+03 1.8e+03

In [23]:
'''
    Writing out climo. files

    All writes are built as delayed tasks and submitted together via
    dask.compute(*...) so monthly + seasonal + annual all stream to disk
    in parallel from one scheduling pass.

    Compression:  zlib level 1 + shuffle typically shrinks CAM climo files
    3–5× with negligible CPU overhead (compression < disk write time).
'''
import dask

if not os.path.exists(dir_out):
    os.makedirs(dir_out)

def _nc_encoding(ds, complevel=1):
    """Default compression encoding for all float data vars in `ds`."""
    enc = {}
    for v in ds.data_vars:
        if np.issubdtype(ds[v].dtype, np.floating):
            enc[v] = {'zlib': True, 'complevel': complevel, 'shuffle': True}
    return enc

delayed_writes = []

# ── Monthly ────────────────────────────────────────────────────────────────
print('-Queuing monthly climatologies...')
for imm, mnum in enumerate(ds_cmonth.month.values):
    fout = dir_out + run_name + '_' + mname_file[imm] + '_climo.nc'
    print(f'  {mname_file[imm]} -- Queuing - {fout}')
    one = ds_cmonth.sel(month=mnum)
    delayed_writes.append(
        one.to_netcdf(fout, mode='w', compute=False, encoding=_nc_encoding(one))
    )

# ── Seasonal (restore attrs wiped by scalar multiplication) ───────────────
print('-Queuing seasonal climatologies...')
ds_wcseas.attrs = ds_vars.attrs
for v in ds_wcseas.data_vars:
    if v in ds_vars.data_vars:
        ds_wcseas[v].attrs = ds_vars[v].attrs
for sname in ds_wcseas.season.values:
    fout = dir_out + run_name + '_' + str(sname) + '_climo.nc'
    print(f'  {sname} -- Queuing - {fout}')
    one = ds_wcseas.sel(season=sname).drop_vars('season')
    delayed_writes.append(
        one.to_netcdf(fout, mode='w', compute=False, encoding=_nc_encoding(one))
    )

# ── Annual ─────────────────────────────────────────────────────────────────
fout_ann = dir_out + run_name + '_ANN_climo.nc'
print(f'-Queuing annual - {fout_ann}')
delayed_writes.append(
    ds_cyear.to_netcdf(fout_ann, mode='w', compute=False, encoding=_nc_encoding(ds_cyear))
)

# ── Fire them all off together ────────────────────────────────────────────
print(f'-Computing & writing {len(delayed_writes)} files in parallel...')
dask.compute(*delayed_writes)
print('---- COMPLETE ----')

-Queuing monthly climatologies...
  01 -- Queuing - /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/climo/f.e30.FLTHIST.CAM7.L48.000a_01_climo.nc
  02 -- Queuing - /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/climo/f.e30.FLTHIST.CAM7.L48.000a_02_climo.nc
  03 -- Queuing - /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/climo/f.e30.FLTHIST.CAM7.L48.000a_03_climo.nc
  04 -- Queuing - /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/climo/f.e30.FLTHIST.CAM7.L48.000a_04_climo.nc
  05 -- Queuing - /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/climo/f.e30.FLTHIST.CAM7.L48.000a_05_climo.nc
  06 -- Queuing - /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/climo/f.e30.FLTHIST.CAM7.L48.000a_06_climo.nc
  07 -- Queuing - /glade/derecho/scratch/rneale/archive/f.e30.FLTHIST.CAM7.L48.000a/climo/f.e30.FLTHIST.CAM7.L48.000a_07_climo.nc
  08 -- Queuing - /glade/derecho/scratch/rneale/archive/